# Prédiction jour de semaine/weekend 

A partir d'une seule ligne, cad d'une suite de valeurs de variables à un instant T dans l'année 2018, le modèle peut déduire si il s'agit d'un weekend ou d'un jour de semaine. 

Il serait possible de donner en entrée toutes les valeurs d'un jour dans l'année et prédire en fonction de la suite d'instants si c'était un weekend ou pas.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.neighbors import (NearestNeighbors, NeighborhoodComponentsAnalysis, KNeighborsClassifier)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Racine du projet — 2 niveaux au-dessus de ce notebook
# notebooks/03_visualisation/ -> notebooks/ -> FlexiMax/
ROOT = Path().resolve().parent.parent

DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'
FILE = "100276-0.parquet"
import traceback

try:
    df = pd.read_parquet(DATA_RAW / FILE)
except Exception:
    traceback.print_exc()

print('Racine projet :', ROOT)
df = pd.read_parquet(DATA_RAW / FILE )

Racine projet : C:\Users\eassoumou1\FlexiMax


In [2]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["day"] = df["timestamp"].dt.day_name()
df

,bldg_id,timestamp,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,...,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.peak_period,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.pre_peak_period,out.schedules.vacancy,day
0,100276,2018-01-01 00:15:00,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,0.680,0.364,0.0,NaN,0.0,Monday
1,100276,2018-01-01 00:30:00,1138.0,0.00216,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,0.644,0.260,0.0,NaN,0.0,Monday
2,100276,2018-01-01 00:45:00,1138.0,0.00240,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,0.621,0.190,0.0,NaN,0.0,Monday
3,100276,2018-01-01 01:00:00,1138.0,0.00204,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,0.637,0.210,0.0,NaN,0.0,Monday
4,100276,2018-01-01 01:15:00,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,0.648,0.224,0.0,NaN,0.0,Monday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,100276,2018-12-31 23:00:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,0.950,0.864,0.0,NaN,0.0,Monday
35036,100276,2018-12-31 23:15:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,0.934,0.936,0.0,NaN,0.0,Monday
35037,100276,2018-12-31 23:30:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,0.934,0.936,0.0,NaN,0.0,Monday
35038,100276,2018-12-31 23:45:00,1138.0,0.00072,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,NaN,0.845,0.715,0.0,NaN,0.0,Monday


In [3]:
df['day']

0         Monday
1         Monday
2         Monday
3         Monday
4         Monday
          ...   
35035     Monday
35036     Monday
35037     Monday
35038     Monday
35039    Tuesday
Name: day, Length: 35040, dtype: str

In [4]:
df['is.weekend'] = df['day'].isin(['Saturday', 'Sunday']).astype(int)
df

,bldg_id,timestamp,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,...,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.peak_period,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.pre_peak_period,out.schedules.vacancy,day,is.weekend
0,100276,2018-01-01 00:15:00,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.680,0.364,0.0,NaN,0.0,Monday,0
1,100276,2018-01-01 00:30:00,1138.0,0.00216,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.644,0.260,0.0,NaN,0.0,Monday,0
2,100276,2018-01-01 00:45:00,1138.0,0.00240,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.621,0.190,0.0,NaN,0.0,Monday,0
3,100276,2018-01-01 01:00:00,1138.0,0.00204,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.637,0.210,0.0,NaN,0.0,Monday,0
4,100276,2018-01-01 01:15:00,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.648,0.224,0.0,NaN,0.0,Monday,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,100276,2018-12-31 23:00:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.950,0.864,0.0,NaN,0.0,Monday,0
35036,100276,2018-12-31 23:15:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.934,0.936,0.0,NaN,0.0,Monday,0
35037,100276,2018-12-31 23:30:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.934,0.936,0.0,NaN,0.0,Monday,0
35038,100276,2018-12-31 23:45:00,1138.0,0.00072,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,NaN,0.845,0.715,0.0,NaN,0.0,Monday,0


In [5]:
y = df['is.weekend'].to_frame()
X = df.select_dtypes(include='number')
X = X.drop(columns=['is.weekend'])
X = X.dropna(axis=1)
X

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy
0,100276,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.260,0.0,0.0,1.0,0.680,0.364,0.0,0.0
1,100276,1138.0,0.00216,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.197,0.0,0.0,1.0,0.644,0.260,0.0,0.0
2,100276,1138.0,0.00240,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.155,0.0,0.0,1.0,0.621,0.190,0.0,0.0
3,100276,1138.0,0.00204,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.1590,0.200,0.0,0.0,1.0,0.637,0.210,0.0,0.0
4,100276,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0177,0.230,0.0,0.0,1.0,0.648,0.224,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,100276,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2390,0.841,0.0,0.0,1.0,0.950,0.864,0.0,0.0
35036,100276,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.764,0.0,0.0,1.0,0.934,0.936,0.0,0.0
35037,100276,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0544,0.764,0.0,0.0,1.0,0.934,0.936,0.0,0.0
35038,100276,1138.0,0.00072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2180,0.593,0.0,0.0,1.0,0.845,0.715,0.0,0.0


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (24528, 179)
X_test shape: (10512, 179)
y_train shape: (24528, 1)
y_test shape: (10512, 1)


In [7]:
X_test

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy
14558,100276,1138.0,0.00300,0.0,0.0,0.04949,0.0,0.00000,0.0,0.0,...,0.000,0.0685,0.1610,0.0,0.0,0.40,0.554,0.0870,0.0,0.0
5878,100276,1138.0,0.00300,0.0,0.0,0.00000,0.0,0.00000,0.0,0.0,...,0.000,0.0000,0.0494,0.0,0.0,0.80,0.562,0.0181,0.0,0.0
16443,100276,1138.0,0.00465,0.0,0.0,0.02917,0.0,0.00000,0.0,0.0,...,0.000,0.0685,0.0693,0.0,0.0,0.88,0.523,0.0232,0.0,0.0
22332,100276,1138.0,0.00300,0.0,0.0,0.06118,0.0,0.00000,0.0,0.0,...,0.000,0.0685,0.1050,0.0,0.0,0.40,0.557,0.0501,0.0,0.0
8887,100276,1138.0,0.00300,0.0,0.0,0.00576,0.0,0.08256,0.0,0.0,...,0.869,0.2060,0.1170,0.0,0.0,0.20,0.579,0.0454,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19372,100276,1138.0,0.00300,0.0,0.0,0.09025,0.0,0.00000,0.0,0.0,...,0.000,0.2060,0.1700,0.0,0.0,0.40,0.593,0.2030,0.0,0.0
20021,100276,1138.0,0.00300,0.0,0.0,0.05916,0.0,0.00000,0.0,0.0,...,0.000,0.0000,0.2210,0.0,0.0,0.60,0.547,0.1740,0.0,0.0
21696,100276,1138.0,0.00448,0.0,0.0,0.07430,0.0,0.00000,0.0,0.0,...,0.000,0.0000,0.2340,0.0,0.0,1.00,0.544,0.1730,0.0,0.0
12071,100276,1138.0,0.00300,0.0,0.0,0.06349,0.0,0.00000,0.0,0.0,...,0.000,0.0000,0.2940,0.0,0.0,0.80,0.680,0.3560,0.0,0.0


In [8]:
y_test

,is.weekend
14558,0
5878,1
16443,0
22332,0
8887,0
...,...
19372,1
20021,1
21696,0
12071,1


KNN

In [9]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print("Sample Predictions:", y_pred[:10])
print("Actual Labels:   ", y_test[:10])

c:\Users\eassoumou1\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\neighbors\_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Sample Predictions: [0 1 0 0 0 1 0 1 1 0]
Actual Labels:           is.weekend
14558           0
5878            1
16443           0
22332           0
8887            0
6702            1
31596           0
7890            1
5210            1
6380            0


In [10]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

Model Accuracy: 0.8333


In [11]:
df_old = df.copy(deep=True)

In [12]:
from random import randint
def random_row_X_test():
    return randint(0,X_test.shape[0])

In [13]:
def predict_if_weekend_KNN(row_index : int):
    
    ROW = X_test.iloc[row_index].to_frame().T
    ROW = ROW.select_dtypes(include='number').dropna(axis=1)
    PREDICTION = knn.predict(ROW)
    
    if PREDICTION == [0]:
        print("Le modèle KNN prédit que ce jour n'est pas un weekend.")
    else:
        print("Le modèle KNN prédit que ce jour est un weekend.")
    
    Xtest_to_df = X_test.iloc[row_index].to_frame().columns[0]
    REALITY = [df['is.weekend'].iloc[Xtest_to_df]]
    if PREDICTION == REALITY :
        print("Le modèle a raison !")
    else:
        print("Le modèle s'est trompé.")
    return PREDICTION

In [14]:
ROW = random_row_X_test()
print("La ligne choisie aléatoirement parmis celles de X_test est la ligne "+str(ROW)+", ce qui correspond à la ligne "+str(X_test.iloc[ROW].to_frame().columns[0])+" de df.")
predict_if_weekend_KNN(ROW)

La ligne choisie aléatoirement parmis celles de X_test est la ligne 2196, ce qui correspond à la ligne 23307 de df.
Le modèle KNN prédit que ce jour est un weekend.
Le modèle s'est trompé.


array([1])

In [15]:
predict_if_weekend_KNN(1211)

Le modèle KNN prédit que ce jour est un weekend.
Le modèle a raison !


array([1])

Gaussian Naïve Bayes

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

c:\Users\eassoumou1\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [17]:
y_pred

array([0, 1, 0, ..., 0, 1, 1], shape=(17520,))

In [18]:
X_test

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy
5063,100276,1138.0,0.00240,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,0.0,0.0685,0.1290,0.0,0.0,0.2,0.619,0.0626,0.0,0.0
35039,100276,1138.0,0.00156,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.3470,0.0,0.0,1.0,0.722,0.4460,0.0,0.0
22416,100276,1138.0,0.00300,0.0,0.0,0.05713,0.0,0.0,0.0,0.0,...,0.0,0.0975,0.1730,0.0,0.0,0.6,0.552,0.0661,0.0,0.0
13063,100276,1138.0,0.00300,0.0,0.0,0.02324,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.0494,0.0,0.0,1.0,0.562,0.0159,0.0,0.0
32985,100276,1138.0,0.00120,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,0.0,0.0685,0.2890,0.0,0.0,0.6,0.715,0.1180,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27082,100276,1138.0,0.00300,0.0,0.0,0.02738,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.0494,0.0,0.0,1.0,0.562,0.0159,0.0,0.0
5576,100276,1138.0,0.00120,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,...,0.0,0.0670,0.1690,0.0,0.0,1.0,0.660,0.1450,0.0,0.0
28288,100276,1138.0,0.00300,0.0,0.0,0.03056,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.1810,0.0,0.0,0.4,0.596,0.0811,0.0,0.0
27935,100276,1138.0,0.00547,0.0,0.0,0.05449,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.3960,0.0,0.0,0.8,0.610,0.3160,0.0,0.0


In [19]:
y_test_array = y_test['is.weekend'].to_numpy()
y_test_array == y_pred

array([ True, False,  True, ...,  True, False, False], shape=(17520,))

In [20]:
print("Number of mislabeled points out of a total "+ str(X_test.shape[0])+" points : "+ str((y_test_array != y_pred).sum()))
print("Accuracy = "+ str(1-((y_test_array != y_pred).sum() /X_test.shape[0])))

Number of mislabeled points out of a total 17520 points : 6888
Accuracy = 0.6068493150684932


In [21]:
def predict_if_weekend_GNB(row_index : int):
    
    ROW = X_test.iloc[row_index].to_frame().T
    ROW = ROW.select_dtypes(include='number').dropna(axis=1)
    PREDICTION = gnb.predict(ROW)
    
    if PREDICTION == [0]:
        print("Le modèle GNB prédit que ce jour n'est pas un weekend.")
    else:
        print("Le modèle GNB prédit que ce jour est un weekend.")
    
    Xtest_to_df = X_test.iloc[row_index].to_frame().columns[0]
    REALITY = [df['is.weekend'].iloc[Xtest_to_df]]
    if PREDICTION == REALITY :
        print("Le modèle a raison !")
    else:
        print("Le modèle s'est trompé.")
    return PREDICTION

In [22]:
ROW = random_row_X_test()
print("La ligne choisie aléatoirement parmis celles de X_test est la ligne "+str(ROW)+", ce qui correspond à la ligne "+str(X_test.iloc[ROW].to_frame().columns[0])+" de df.")
predict_if_weekend_GNB(ROW)

La ligne choisie aléatoirement parmis celles de X_test est la ligne 2482, ce qui correspond à la ligne 18750 de df.
Le modèle GNB prédit que ce jour est un weekend.
Le modèle a raison !


array([1])

# Prédiction du jour de la semaine

Le modèle de prédiction a toutes les données pour un jour en entrée, et les 7 options différentes (une par jour de la semaine) en soirtie.

In [23]:
df["hour"] = df["timestamp"].dt.hour
df["month"] = df["timestamp"].dt.month
df["date"] = df["timestamp"].dt.date

In [24]:
df['day.of.year']=0
dict = {date: i for i, date in enumerate(df["date"].unique())}
df["day.of.year"] = df["date"].map(dict)
df

,bldg_id,timestamp,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,...,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.pre_peak_period,out.schedules.vacancy,day,is.weekend,hour,month,date,day.of.year
0,100276,2018-01-01 00:15:00,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,...,0.364,0.0,NaN,0.0,Monday,0,0,1,2018-01-01,0
1,100276,2018-01-01 00:30:00,1138.0,0.00216,0.0,0.0,0.0,0.0,0.0,0.0,...,0.260,0.0,NaN,0.0,Monday,0,0,1,2018-01-01,0
2,100276,2018-01-01 00:45:00,1138.0,0.00240,0.0,0.0,0.0,0.0,0.0,0.0,...,0.190,0.0,NaN,0.0,Monday,0,0,1,2018-01-01,0
3,100276,2018-01-01 01:00:00,1138.0,0.00204,0.0,0.0,0.0,0.0,0.0,0.0,...,0.210,0.0,NaN,0.0,Monday,0,1,1,2018-01-01,0
4,100276,2018-01-01 01:15:00,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,...,0.224,0.0,NaN,0.0,Monday,0,1,1,2018-01-01,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,100276,2018-12-31 23:00:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.864,0.0,NaN,0.0,Monday,0,23,12,2018-12-31,364
35036,100276,2018-12-31 23:15:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.936,0.0,NaN,0.0,Monday,0,23,12,2018-12-31,364
35037,100276,2018-12-31 23:30:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.936,0.0,NaN,0.0,Monday,0,23,12,2018-12-31,364
35038,100276,2018-12-31 23:45:00,1138.0,0.00072,0.0,0.0,0.0,0.0,0.0,0.0,...,0.715,0.0,NaN,0.0,Monday,0,23,12,2018-12-31,364


In [25]:
dict_1 = {day: i for i, day in enumerate(df["day"].unique())}
dict_1

{'Monday': 0,
 'Tuesday': 1,
 'Wednesday': 2,
 'Thursday': 3,
 'Friday': 4,
 'Saturday': 5,
 'Sunday': 6}

In [26]:
df['day.of.week'] = df['day'].map(dict_1)
#df_days = df_num.groupby("day.of.year")[df_num.columns].mean()
df

,bldg_id,timestamp,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,...,out.schedules.power_outage,out.schedules.pre_peak_period,out.schedules.vacancy,day,is.weekend,hour,month,date,day.of.year,day.of.week
0,100276,2018-01-01 00:15:00,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,0.0,Monday,0,0,1,2018-01-01,0,0
1,100276,2018-01-01 00:30:00,1138.0,0.00216,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,0.0,Monday,0,0,1,2018-01-01,0,0
2,100276,2018-01-01 00:45:00,1138.0,0.00240,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,0.0,Monday,0,0,1,2018-01-01,0,0
3,100276,2018-01-01 01:00:00,1138.0,0.00204,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,0.0,Monday,0,1,1,2018-01-01,0,0
4,100276,2018-01-01 01:15:00,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,0.0,Monday,0,1,1,2018-01-01,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,100276,2018-12-31 23:00:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,0.0,Monday,0,23,12,2018-12-31,364,0
35036,100276,2018-12-31 23:15:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,0.0,Monday,0,23,12,2018-12-31,364,0
35037,100276,2018-12-31 23:30:00,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,0.0,Monday,0,23,12,2018-12-31,364,0
35038,100276,2018-12-31 23:45:00,1138.0,0.00072,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,0.0,Monday,0,23,12,2018-12-31,364,0


            Idee d'algorithme : 
            
            - Utiliser le modèles qui sait identifier les weekends pour trouver quel jour est le LUNDI (0): celui ou les deux jours de la semaine d'avant sont des weekends.

            - Pour cela, il faut un moyen de regroupper de manière ordonnée les jours. Il faut trouver le nombre de lignes par jour : utilser un loc par day of year pour avoir un dataframe pour un day of year spécifique. En prenant une des lignes (par ex au milieu), on peut appliquer le KNN pour voir si c'est un jour de weekend ou pas et trouver l lundi par une fonction. 

            - Ensuite on va faire une fonction qui calcule la distance entre elle et le lundi d'avant (comptant les indices dans la table dataframe avec le groupby day of year). Cette distance nous donne directement le jour de la semaine actuel par correspondance avec dict_1. 

            - Si il n'y a pas de lundi d'avant (on peut vérifier que le day of year actuel - 6 est positif ou nul, sinon on sait qu'on est dans la 1ere semaine de juillet). Dans ce cas, on prend le lundi d'après, mais alors le jour de la semaine devra se trouver en prenant 7-distance si on n'est pas le lundi, et 0 si on est déjà sur le lundi. 

In [27]:
print("Il y a "+ str(df.loc[df["day.of.year"]==1].shape[0])+" lignes enregistrées par jour (sauf le 1er janvier ou c'est réparti sur 2018 et 2019).")

Il y a 96 lignes enregistrées par jour (sauf le 1er janvier ou c'est réparti sur 2018 et 2019).


In [28]:
df_num = df.select_dtypes(include='number').dropna(axis=1)
df_test = df_num.drop(columns = ['is.weekend','day.of.year','day.of.week','hour','month'])
df_test

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy
0,100276,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.260,0.0,0.0,1.0,0.680,0.364,0.0,0.0
1,100276,1138.0,0.00216,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.197,0.0,0.0,1.0,0.644,0.260,0.0,0.0
2,100276,1138.0,0.00240,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.155,0.0,0.0,1.0,0.621,0.190,0.0,0.0
3,100276,1138.0,0.00204,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.1590,0.200,0.0,0.0,1.0,0.637,0.210,0.0,0.0
4,100276,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0177,0.230,0.0,0.0,1.0,0.648,0.224,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,100276,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2390,0.841,0.0,0.0,1.0,0.950,0.864,0.0,0.0
35036,100276,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.764,0.0,0.0,1.0,0.934,0.936,0.0,0.0
35037,100276,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0544,0.764,0.0,0.0,1.0,0.934,0.936,0.0,0.0
35038,100276,1138.0,0.00072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2180,0.593,0.0,0.0,1.0,0.845,0.715,0.0,0.0


In [29]:
ROW = df_test.loc[df["day.of.year"]==35].iloc[48]
ROW = ROW.to_frame().T
ROW

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy
3407,100276.0,1138.0,0.00108,0.0,0.0,0.00694,0.0,0.0,0.0,0.0,...,0.0,0.165,0.324,0.0,0.0,0.8,0.729,0.097,0.0,0.0


In [30]:
knn.predict(ROW)

array([0])

In [31]:
df_test.loc[df["day.of.year"]==35].iloc[48]

bldg_id                                                   100276.00000
in.sqft                                                     1138.00000
out.electricity.ceiling_fan.energy_consumption..kwh            0.00108
out.electricity.clothes_dryer.energy_consumption..kwh          0.00000
out.electricity.clothes_washer.energy_consumption..kwh         0.00000
                                                              ...     
out.schedules.occupants                                        0.80000
out.schedules.plug_loads_other                                 0.72900
out.schedules.plug_loads_tv                                    0.09700
out.schedules.power_outage                                     0.00000
out.schedules.vacancy                                          0.00000
Name: 3407, Length: 179, dtype: float64

In [32]:
df_test.iloc[780]

bldg_id                                                   100276.0000
in.sqft                                                     1138.0000
out.electricity.ceiling_fan.energy_consumption..kwh            0.0024
out.electricity.clothes_dryer.energy_consumption..kwh          0.0000
out.electricity.clothes_washer.energy_consumption..kwh         0.0000
                                                             ...     
out.schedules.occupants                                        1.0000
out.schedules.plug_loads_other                                 0.5890
out.schedules.plug_loads_tv                                    0.0318
out.schedules.power_outage                                     0.0000
out.schedules.vacancy                                          0.0000
Name: 780, Length: 179, dtype: float64

In [33]:
#Choix de l'heure de la journée selon laquelle le modèle va prédire weekend/jour de la semaine
HEURE_CHOISIE = 7

In [34]:
def advanced_n_days_ROW(current_row_index,n):
    skip = n*96
    day_of_year = current_row_index+skip
    ROW = df_test.iloc[HEURE_CHOISIE+day_of_year]
    ROW = ROW.to_frame().T
    return (ROW, day_of_year)

In [35]:
print(advanced_n_days_ROW(35,1)[1])
advanced_n_days_ROW(35,1)[0]

131


,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy
138,100276.0,1138.0,0.0018,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0685,0.209,0.0,0.0,1.0,0.656,0.0508,0.0,0.0


In [36]:
print(advanced_n_days_ROW(35,2)[1])
advanced_n_days_ROW(35,2)[0]

227


,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy
234,100276.0,1138.0,0.00072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.225,0.352,0.0,0.0,0.76,0.741,0.0822,0.0,0.0


In [37]:
knn.predict(advanced_n_days_ROW(500,-2)[0])

array([0])

In [38]:
df_test

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy
0,100276,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.260,0.0,0.0,1.0,0.680,0.364,0.0,0.0
1,100276,1138.0,0.00216,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.197,0.0,0.0,1.0,0.644,0.260,0.0,0.0
2,100276,1138.0,0.00240,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.155,0.0,0.0,1.0,0.621,0.190,0.0,0.0
3,100276,1138.0,0.00204,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.1590,0.200,0.0,0.0,1.0,0.637,0.210,0.0,0.0
4,100276,1138.0,0.00180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0177,0.230,0.0,0.0,1.0,0.648,0.224,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,100276,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2390,0.841,0.0,0.0,1.0,0.950,0.864,0.0,0.0
35036,100276,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0000,0.764,0.0,0.0,1.0,0.934,0.936,0.0,0.0
35037,100276,1138.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0544,0.764,0.0,0.0,1.0,0.934,0.936,0.0,0.0
35038,100276,1138.0,0.00072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2180,0.593,0.0,0.0,1.0,0.845,0.715,0.0,0.0


In [39]:
df_days = df_num.groupby("day.of.year")[df_num.columns].mean()
df_days

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy,is.weekend,hour,month,day.of.year,day.of.week
day.of.year,,,,,,,,,,,,,,,,,,,,,
0,100276.0,1138.0,0.001972,0.013680,0.001953,0.000000,0.0,0.004693,0.0,0.0,...,0.696842,0.665947,0.150641,0.0,0.0,0.0,11.621053,1.0,0.0,0.0
1,100276.0,1138.0,0.001469,0.054986,0.002865,0.006892,0.0,0.010206,0.0,0.0,...,0.816667,0.707656,0.199393,0.0,0.0,0.0,11.500000,1.0,1.0,1.0
2,100276.0,1138.0,0.001706,0.041449,0.000933,0.004080,0.0,0.004644,0.0,0.0,...,0.725000,0.690135,0.195155,0.0,0.0,0.0,11.500000,1.0,2.0,2.0
3,100276.0,1138.0,0.002120,0.040612,0.001866,0.009524,0.0,0.000000,0.0,0.0,...,0.668750,0.654604,0.157768,0.0,0.0,0.0,11.500000,1.0,3.0,3.0
4,100276.0,1138.0,0.002037,0.041449,0.001933,0.013014,0.0,0.010206,0.0,0.0,...,0.762500,0.665948,0.149703,0.0,0.0,0.0,11.500000,1.0,4.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,100276.0,1138.0,0.001862,0.027912,0.002932,0.003145,0.0,0.004702,0.0,0.0,...,0.725000,0.678927,0.175503,0.0,0.0,0.0,11.500000,12.0,361.0,4.0
362,100276.0,1138.0,0.001622,0.027912,0.000933,0.001513,0.0,0.010320,0.0,0.0,...,0.857083,0.693385,0.222695,0.0,0.0,1.0,11.500000,12.0,362.0,5.0
363,100276.0,1138.0,0.001950,0.000000,0.000000,0.003117,0.0,0.000000,0.0,0.0,...,0.791667,0.672802,0.201264,0.0,0.0,1.0,11.500000,12.0,363.0,6.0


In [40]:
knn.predict(advanced_n_days_ROW(100,-1)[0])

array([0])

In [41]:
knn.predict(advanced_n_days_ROW(500,0)[0])

array([1])

In [42]:
knn.predict(advanced_n_days_ROW(500,1)[0])

array([1])

In [43]:
knn.predict(advanced_n_days_ROW(500,2)[0])

array([0])

            --> Fonction qui trouve le lundi 

In [44]:
def find_monday(current_row_index):
    #if df_num["day.of.year"].iloc[current_row_index] - 6 >= 0:
        week_prediction_table = []
        dist_to_Monday = 0
        Monday_index = 0
        for i in range(7):
            ROW_to_predict = advanced_n_days_ROW(current_row_index,i)[0]
            week_prediction_table.append(knn.predict(ROW_to_predict))
        for i in range(len(week_prediction_table)-2):
            if week_prediction_table[i]==[1] and week_prediction_table[i+1]==[1] and week_prediction_table[i+2]==[0]:
                dist_to_Monday = i+2
            elif week_prediction_table[5]==1 and week_prediction_table[6]==1 and week_prediction_table[0]==0:
                dist_to_Monday = 0
            elif week_prediction_table[6]==1 and week_prediction_table[0]==1 and week_prediction_table[1]==0:
                dist_to_Monday = 1
            elif week_prediction_table[i]==[1] and week_prediction_table[i+1]==[0]:
                dist_to_Monday = i+1
            elif week_prediction_table[6]==[1] and week_prediction_table[0]==[0]:
                dist_to_Monday = 0
            Monday_index  = dist_to_Monday*96 + current_row_index
        return (dist_to_Monday, week_prediction_table, Monday_index)

#coller les deux fonctions 
#trouver l'indice du monday dans df_test en y additionnant les indices etc

In [45]:
def find_monday(current_row_index):
    #if df_num["day.of.year"].iloc[current_row_index] - 6 >= 0:
        week_prediction_table = []
        dist_to_Monday = 0
        Monday_index = 0
        for i in range(7):
                ROW_to_predict = advanced_n_days_ROW(current_row_index,i)[0]
                week_prediction_table.append(knn.predict(ROW_to_predict))
        for i in range(len(week_prediction_table)-1):
            if week_prediction_table[i]==[1] and week_prediction_table[i+1]==[0]:
                dist_to_Monday = i+1
            elif week_prediction_table[6]==[1] and week_prediction_table[0]==[0]:
                dist_to_Monday = 0
            Monday_index  = dist_to_Monday*96 + current_row_index
        return (dist_to_Monday, week_prediction_table, Monday_index)

In [46]:
def check_monday(result):
    line_found = df_num.iloc[result[2]].to_frame().T
    real_day = line_found["day.of.week"].values[0]
    if real_day==0:
        return True
    else:
        print(real_day)
        return False

In [47]:
df_num["day.of.year"].iloc[200] - 6 

np.int64(-4)

In [48]:
find_monday(848)

(6,
 [array([0]),
  array([0]),
  array([0]),
  array([0]),
  array([1]),
  array([1]),
  array([0])],
 1424)

In [49]:
def accuracy_test(iterations):
    box = []
    true_count = 0
    max  = df_test.shape[0] - (HEURE_CHOISIE+6*96)
    for i in range(iterations):
        rand = randint(0, max)
        box.append(check_monday(find_monday(rand)))
    for e in box:
        if e==True:
            true_count+=1
    print ("Accuracy : "+ str(true_count/len(box)))
    return box

In [147]:
accuracy_test(200)

1.0
6.0
1.0
6.0
2.0
6.0
5.0
4.0
6.0
6.0
2.0
6.0
6.0
2.0
2.0
6.0
2.0
3.0
6.0
6.0
6.0
3.0
6.0
6.0
5.0
5.0
2.0
1.0
1.0
2.0
6.0
6.0
6.0
6.0
1.0
6.0
4.0
2.0
4.0
5.0
1.0
3.0
4.0
4.0
2.0
5.0
6.0
6.0
6.0
6.0
1.0
4.0
6.0
6.0
3.0
6.0
6.0
1.0
6.0
5.0
4.0
5.0
6.0
3.0
6.0
3.0
6.0
6.0
6.0
6.0
2.0
5.0
Accuracy : 0.64


[False,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 False,
 True,
 True,
 True,
 False,
 True,
 True,
 False,
 False,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 True,
 True,
 True,
 False,
 False,
 True,
 False,
 True,
 True,
 False,
 True,
 False,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 False,
 False,
 True,
 True,
 True,
 False,
 True,
 False,
 True,
 True,
 False,
 False,
 True,
 False,
 True,
 False,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 True,
 True,
 False,
 True,
 True,
 False,
 True,
 False,
 False,
 True,
 False,
 True,
 True,
 False,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 False,
 Fa

In [148]:
accuracy_test(1000)

6.0
6.0
5.0
4.0
6.0
6.0
6.0
1.0
5.0
3.0
5.0
2.0
3.0
6.0
5.0
4.0
6.0
6.0
2.0
4.0
6.0
4.0
2.0
3.0
1.0
6.0
6.0
6.0
2.0
5.0
6.0
6.0
6.0
2.0
3.0
4.0
6.0
2.0
6.0
6.0
2.0
3.0
4.0
6.0
4.0
5.0
4.0
6.0
1.0
1.0
5.0
6.0
6.0
5.0
1.0
6.0
6.0
3.0
6.0
4.0
3.0
6.0
6.0
3.0
3.0
2.0
5.0
6.0
6.0
3.0
1.0
5.0
3.0
1.0
3.0
5.0
4.0
6.0
6.0
6.0
6.0
6.0
2.0
1.0
3.0
6.0
3.0
4.0
6.0
6.0
6.0
2.0
4.0
4.0
1.0
2.0
6.0
6.0
6.0
6.0
1.0
4.0
2.0
4.0
2.0
4.0
6.0
6.0
2.0
2.0
6.0
2.0
1.0
6.0
6.0
6.0
3.0
3.0
6.0
4.0
2.0
6.0
5.0
6.0
6.0
6.0
6.0
5.0
4.0
6.0
6.0
2.0
6.0
6.0
6.0
2.0
1.0
4.0
6.0
4.0
6.0
6.0
5.0
6.0
6.0
1.0
5.0
6.0
6.0
6.0
6.0
6.0
2.0
6.0
3.0
1.0
1.0
6.0
6.0
1.0
1.0
6.0
2.0
6.0
5.0
6.0
3.0
1.0
6.0
2.0
4.0
1.0
3.0
5.0
6.0
6.0
6.0
1.0
5.0
6.0
6.0
6.0
1.0
3.0
1.0
6.0
6.0
2.0
6.0
1.0
6.0
6.0
6.0
1.0
5.0
6.0
6.0
1.0
1.0
1.0
6.0
1.0
4.0
4.0
6.0
6.0
1.0
1.0
6.0
6.0
1.0
6.0
6.0
2.0
6.0
2.0
6.0
4.0
6.0
6.0
6.0
1.0
6.0
4.0
6.0
3.0
6.0
6.0
6.0
4.0
6.0
6.0
6.0
6.0
6.0
5.0
5.0
1.0
6.0
6.0
6.0
4.0
1.0
3.0
2.0
6.0
6.0
1.0
6.0
6.0


[True,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 False,
 False,
 False,
 True,
 True,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 True,
 True,
 False,
 False,
 True,
 True,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 False,
 False,
 False,
 True,
 True,
 True,
 False,
 False,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 False,
 True,
 False,
 True,
 True,
 True,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 True,
 False,
 False,
 True,
 True,
 False,
 False,
 True,
 True,


In [ ]:
def find_current_day(threevals):
    Monday = threevals[0]
    Day = 7 - Monday
    for key,val in dict_1.items():
        if val == Day:
            return key

In [53]:
check_monday(find_monday(196))

True

In [54]:
find_current_day(find_monday(196))

'Wednesday'

In [56]:
dict_2 = {val:cle for cle, val in dict_1.items()}
dict_2

{0: 'Monday',
 1: 'Tuesday',
 2: 'Wednesday',
 3: 'Thursday',
 4: 'Friday',
 5: 'Saturday',
 6: 'Sunday'}

In [58]:
def check_if_current_day(index):
    prediction = find_current_day(find_monday(index))
    reality = dict_2[df["day.of.week"].iloc[index]]
    if prediction==reality:
        return True
    else:
        return False 

def precise_day_accuracy_test(iterations):
    box = []
    true_count = 0
    max  = df_test.shape[0] - (HEURE_CHOISIE+6*96)
    for i in range(iterations):
        rand = randint(0,max)
        box.append(check_if_current_day(rand))
    for e in box :
        if e==True:
            true_count+=1
    print("Accuracy :"+str(true_count/iterations))

In [61]:
precise_day_accuracy_test(1000)

Accuracy :0.493


### Tests

In [120]:
def visualizer(selected_row):
    for i in range(7):
        c=selected_row+i*96
        line_found = df_num.iloc[c].to_frame().T
        real_day = line_found["day.of.week"].values[0]
        ROW_to_predict = advanced_n_days_ROW(c,0)[0]
        x=knn.predict(ROW_to_predict)
        print(c, real_day,x)
    return

In [121]:
find_monday(25669)

(5,
 [array([0]),
  array([0]),
  array([0]),
  array([0]),
  array([1]),
  array([0]),
  array([0])],
 26149)

In [122]:
check_monday(find_monday(25669))

6.0


False

In [123]:
visualizer(32741)

32741 5.0 [1]
32837 6.0 [1]
32933 0.0 [0]
33029 1.0 [0]
33125 2.0 [0]
33221 3.0 [0]
33317 4.0 [0]


In [124]:
df_num.iloc[848].to_frame().T

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy,is.weekend,hour,month,day.of.year,day.of.week
848,100276.0,1138.0,0.0018,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.4,0.726,0.17,0.0,0.0,0.0,20.0,1.0,8.0,1.0


In [125]:
visualizer(200)

200 2.0 [0]
296 3.0 [0]
392 4.0 [0]
488 5.0 [0]
584 6.0 [1]
680 0.0 [0]
776 1.0 [0]


In [126]:
ROW_to_predict = advanced_n_days_ROW(848+4*96,0)[0]
knn.predict(ROW_to_predict)

array([0])

In [127]:
df_num.iloc[1088].to_frame().T

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy,is.weekend,hour,month,day.of.year,day.of.week
1088,100276.0,1138.0,0.00264,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.578,0.0195,0.0,0.0,0.0,8.0,1.0,11.0,4.0


In [128]:
df_num["day.of.year"].iloc[4607] - 6


np.int64(42)

In [129]:
df_num[df_num["day.of.year"]==48]

,bldg_id,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,...,out.schedules.occupants,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.vacancy,is.weekend,hour,month,day.of.year,day.of.week
4607,100276,1138.0,0.00156,0.73078,0.03839,0.01111,0.0,0.00000,0.0,0.0,...,1.0,0.725,0.468,0.0,0.0,1,0,2,48,6
4608,100276,1138.0,0.00180,1.21797,0.00000,0.00000,0.0,0.00000,0.0,0.0,...,1.0,0.682,0.377,0.0,0.0,1,0,2,48,6
4609,100276,1138.0,0.00180,1.21797,0.00000,0.00000,0.0,0.00000,0.0,0.0,...,1.0,0.682,0.377,0.0,0.0,1,0,2,48,6
4610,100276,1138.0,0.00180,0.64918,0.00000,0.00000,0.0,0.00000,0.0,0.0,...,1.0,0.682,0.377,0.0,0.0,1,0,2,48,6
4611,100276,1138.0,0.00180,0.00000,0.00000,0.01668,0.0,0.00000,0.0,0.0,...,1.0,0.663,0.296,0.0,0.0,1,1,2,48,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4698,100276,1138.0,0.00060,0.00000,0.00000,0.03786,0.0,0.08256,0.0,0.0,...,0.8,0.897,0.679,0.0,0.0,1,22,2,48,6
4699,100276,1138.0,0.00060,0.00000,0.00000,0.03810,0.0,0.06052,0.0,0.0,...,0.8,0.878,0.750,0.0,0.0,1,23,2,48,6
4700,100276,1138.0,0.00060,0.00000,0.00000,0.03787,0.0,0.00000,0.0,0.0,...,0.8,0.865,0.798,0.0,0.0,1,23,2,48,6
4701,100276,1138.0,0.00060,0.00000,0.00000,0.03795,0.0,0.00000,0.0,0.0,...,0.8,0.865,0.798,0.0,0.0,1,23,2,48,6


Pour avoir une plus grande accuracy, on peut décaler le moment ou est fait la prédiction. 

    - 12h --> accuracy = 0.3
    - 7h --> accuracy = 0.5

Pour trouver l'heure avec la plus grande accuracy, il faut comparer par exemple les courbes de charge et/ou d'occupation entre le weekeend et la semaine en moyenne. 